In [ ]:
import os
import re
import glob
import pandas as pd
import pyarrow.parquet as pq
from tqdm import tqdm

ROOT = "./droid_1.0.1_first_20_chunks"
NUM_CHUNKS = 20
OUT_CSV = "droid_first_20_chunks_markovian_split.csv"

MARKOVIAN_KEYWORDS = [
    "pick up", "pick", "grasp", "grab", "lift",
    "place", "put", "move", "push", "pull",
    "open", "close", "turn on", "turn off",
    "press", "insert", "remove", "slide",
]

NON_MARKOVIAN_KEYWORDS = [
    "then", "after", "before", "first", "second", "third", "finally", "next",
    "all", "every", "each", "remaining", "another",
    "sort", "arrange", "organize", "clean up", "cleanup",
    "stack", "unstack", "set up", "prepare",
    "repeat", "again", "until",
    "in order", "sequence",
]


def normalize_text(x):
    if x is None:
        return ""
    return str(x).lower().strip()


def classify_task(text):
    text = normalize_text(text)

    non_score = 0
    markov_score = 0

    for kw in NON_MARKOVIAN_KEYWORDS:
        if re.search(rf"\b{re.escape(kw)}\b", text):
            non_score += 2

    for kw in MARKOVIAN_KEYWORDS:
        if re.search(rf"\b{re.escape(kw)}\b", text):
            markov_score += 1

    # Multiple action verbs usually means longer-horizon behavior
    action_count = sum(
        1 for kw in MARKOVIAN_KEYWORDS
        if re.search(rf"\b{re.escape(kw)}\b", text)
    )

    if action_count >= 2 and any(w in text for w in ["then", "after", "before", "and"]):
        non_score += 2

    # Broad household goals are often non-Markovian because progress matters
    broad_goal_words = ["clean", "tidy", "organize", "arrange", "sort"]
    if any(w in text for w in broad_goal_words):
        non_score += 2

    if non_score > markov_score:
        return "non_markovian"

    return "markovian"


def read_episode_summary(parquet_path):
    columns = [
        "episode_index",
        "task_index",
        "language_instruction",
        "language_instruction_2",
        "language_instruction_3",
        "task_category",
        "is_episode_successful",
    ]

    table = pq.read_table(
        parquet_path,
        columns=[c for c in columns if c in pq.read_schema(parquet_path).names],
    )

    df = table.slice(0, 1).to_pandas()
    row = df.iloc[0].to_dict()

    instruction_parts = [
        row.get("language_instruction", ""),
        row.get("language_instruction_2", ""),
        row.get("language_instruction_3", ""),
        row.get("task_category", ""),
    ]

    instruction_text = " | ".join(
        [normalize_text(x) for x in instruction_parts if normalize_text(x)]
    )

    return {
        "episode_index": row.get("episode_index"),
        "task_index": row.get("task_index"),
        "language_instruction": row.get("language_instruction", ""),
        "language_instruction_2": row.get("language_instruction_2", ""),
        "language_instruction_3": row.get("language_instruction_3", ""),
        "task_category": row.get("task_category", ""),
        "is_episode_successful": row.get("is_episode_successful"),
        "instruction_text": instruction_text,
        "label": classify_task(instruction_text),
        "parquet_path": parquet_path,
    }


all_files = []

for i in range(NUM_CHUNKS):
    chunk = f"chunk-{i:03d}"
    pattern = os.path.join(ROOT, "data", chunk, "*.parquet")
    all_files.extend(sorted(glob.glob(pattern)))

print(f"Found {len(all_files)} parquet files")

rows = []

for path in tqdm(all_files):
    try:
        rows.append(read_episode_summary(path))
    except Exception as e:
        rows.append({
            "episode_index": None,
            "task_index": None,
            "instruction_text": "",
            "label": "read_error",
            "parquet_path": path,
            "error": str(e),
        })

df = pd.DataFrame(rows)

df.to_csv(OUT_CSV, index=False)

print(df["label"].value_counts())
print(f"Saved to {OUT_CSV}")